In [2]:
import classifier
from scipy.optimize import curve_fit
import numpy as np

red, green, blue = '#ff0040', '#00aa00', '#187bff'

bistroke_times = {}
bistroke_freq = {}
bigrams = set()

with open(f"nstrokes/bistrokes_0.txt") as file:
    for l in file:
        layout, bigram, freq, *times = l.split("\t")

        if bigram not in bigrams:
            bigrams.add(bigram)
            
        bistroke_freq[(layout, bigram)] = int(freq)
        bistroke_times[(layout, bigram)] = [
            list(map(int, t.strip()[1:-1].split(", "))) for t in times
        ]

In [3]:
layout = "qwerty"

freqs = []
times = []
is_sfb = []
home_row = []
not_homerow = []
delta_x = []
is_bottom = []
top1 = []
home1 = []
bottom1 = []
top2 = []
home2 = []
bottom2 = []
finger1 = []
finger2 = []
same_hand = []
is_pinky1 = []
is_pinky2 = []
is_ring1 = []
is_ring2 = []
is_middle1 = []
is_middle2 = []
is_index1 = []
is_index2 = []
dists = []
y_dist = []
c = []

def get_iqr_avg(data):
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3-Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    
    new_data = [x for x in data if x >= lower_bound and x <= upper_bound]
    print(len(data), len(new_data))
    
    return sum(new_data)/len(new_data)


for bg in bigrams:
    if not any([c in '!@#$%^&*()QWERTYUIOP{}|ASDFGHJKL:"ZXCVBNM<>?' for c in bg]):
        if (layout, bg) in bistroke_freq:
            time_data = [t[1] for t in bistroke_times[(layout, bg)] if t[0] > 47 and t[1] < 1000]
            x_val = bistroke_freq[(layout, bg)]
            y_val = get_iqr_avg(time_data)
            
            if x_val != 0:
                freqs.append(x_val)
                times.append(y_val)
                is_sfb.append(int(classifier.same_finger(bg)))
                delta_x.append(classifier.get_dx(bg))
                top1.append(classifier.kb.get_row(bg[0]) == 3)
                home1.append(classifier.kb.get_row(bg[0]) == 2)
                bottom1.append(classifier.kb.get_row(bg[0]) == 1)
                top2.append(classifier.kb.get_row(bg[1]) == 3)
                home2.append(classifier.kb.get_row(bg[1]) == 2)
                bottom2.append(classifier.kb.get_row(bg[1]) == 1)
                y_dist.append(classifier.get_dy(bg))
                same_hand.append(classifier.same_hand(bg))

                # Finger stuff
                finger1.append(abs(classifier.kb.get_col(bg[0])))
                finger2.append(abs(classifier.kb.get_col(bg[1])))
                
                is_pinky1.append(abs(classifier.kb.get_col(bg[0])) == 5)
                is_pinky2.append(abs(classifier.kb.get_col(bg[1])) == 5)
                is_ring1.append(abs(classifier.kb.get_col(bg[0])) == 4)
                is_ring2.append(abs(classifier.kb.get_col(bg[1])) == 4)
                is_middle1.append(abs(classifier.kb.get_col(bg[0])) == 3)
                is_middle2.append(abs(classifier.kb.get_col(bg[1])) == 3)
                is_index1.append(abs(classifier.kb.get_col(bg[0])) in (1,2))
                is_index2.append(abs(classifier.kb.get_col(bg[1])) in (1,2))

                c.append(red if is_sfb[-1] else blue if same_hand[-1] else green)

71131 67689
13295 12477
3904 3634
318072 299177
315 293
20143 18654
437 411
5686 5241
47680 45124
4512 4054
3324 3188
866 809
14373 13223
331 309
59337 56616
23046 21526
15500 14443
48097 45358
3117 2930
1418 1378
10799 10006
32285 30652
13322 12403
59625 54995
3994 3727
6199 5703
8779 7954
8195 7873
35200 33571
315 296
84747 82294
1526 1409
134687 125506
1181 1064
78292 72670
829 790
10185 9540
48319 44488
16941 15623
10700 9977
10657 10041
2847 2647
310 294
22175 20821
107299 103688
10301 9563
67020 64475
136044 130779
1983 1851
295 277
2669 2433
489913 475166
22458 20499
510 490
594 561
38984 37306
871 848
109858 106940
6836 6287
52021 49754
27710 26571
8173 7512
455 430
11690 11096
4029 3779
45591 44012
61948 58465
60340 55806
9837 9082
11778 10922
75094 70357
33334 31613
462 440
2946 2779
10586 9923
1243 1152
360 346
362 350
392 379
15979 14779
720 706
88087 82902
1330 1233
426 406
50568 48701
2861 2722
434 413
176970 169899
68331 64664
4905 4532
189636 185259
6464 5935
107408 998

In [107]:
# def model_function(x, a, b, c, d, e, f, g, h, i, j, k, l):
#     freq_pen = (a * np.log(x[0] + b) + c)
#     finger_pen = (d*x[1]+e*x[2]+f*x[3]+g*x[4] + h)
#     
#     return freq_pen*finger_pen
#     return (a * np.log(x[0] + b) + c) * (d * x[1] + e * x[2] + g * x[3] + f)

# freqs, is_pinky1, is_ring1, is_middle1, is_index1, is_pinky2, is_ring2, is_middle2, is_index2, is_sfb, delta_x, y_dist, same_hand, bottom2, home2, top2
def model_function(features, p1, p2, p3, p4, p5, p6, p7, p8, p9, p10, p11, p12, p13, p14, p15, p16, p17, p18, p19, p20, p21, p22): #  p67, p68,p70,  p71, p72,
    freq, is_sfb, dx, dy, same_hand = features[0], *features[9:13]
    bottom1, home1, top1, bottom2, home2, top2 = features[13:19]
    is_pinky1, is_ring1, is_middle1, is_index1 = features[1:5]
    is_pinky2, is_ring2, is_middle2, is_index2 = features[5:9]

    # Finger weighting
    freq_pen = (p1 * np.log(freq + p2) + p3)

    first_finger_row_pen = p4*bottom1 + p5*home1 + p6*top1
    second_finger_row_pen = p8*bottom2 + p9*home2 + p10*top2

    first_finger_col_pen = p12*is_pinky1 + p13*is_ring1 + p14*is_middle1 + p15*is_index1
    second_finger_col_pen = p16*is_pinky2 + p17*is_ring2 + p18*is_middle2 + p19*is_index2
    row_pen = (first_finger_row_pen + second_finger_row_pen)
    col_pen = (first_finger_col_pen + second_finger_col_pen)

    # sfb_col_pen = p12*is_pinky1 + p13*is_ring1 + p14*is_middle1 + p15*is_index1
    
    # BG Type Classification
    same_hand_weight = p20*same_hand*(1-is_sfb)
    sfb_weight = p22*is_sfb

    finger_pen = row_pen * col_pen

    return freq_pen + same_hand_weight + sfb_weight + finger_pen

"""def model_function(features, p1, p2, p3, p4, p5, p6, p7, p8, p9, p10, p11, p12, p13, p14, p15, p16, p17, p18, p19, p20, 
                   p21, p22, p23, p24, p25, p26, p27, p28, p29, p30, p31, p32, p33, p34, p35, p36, p37, p38, p39, p40, 
                   p41, p42, p43, p44, p45, p46, p47, p48, p49, p50, p51, p52, p53, p54, p55, p56, p57, p58, p59, p60, 
                   p61, p62, p63, p64, p65, p66, p69, p70, p71, p73, p74, p75, p76, p77, p78, p79): #  p67, p68,p70,  p71, p72,
    freq, is_sfb, dx, dy, same_hand = features[0], *features[9:13]
    bottom1, home1, top1, bottom2, home2, top2 = features[13:19]
    freq_pen = (p1 * np.log(freq + p2) + p3)
    
    row_pen1 = (p4 * bottom2 + p5 * home2 + p6 * top2 + p7)
    row_pen2 = (p8 * bottom2 + p9 * home2 + p10 * top2 + p11)
    row_pen2b = (p12 * bottom1 + p13 * home1 + p14 * top1 + p15)
    row_pen3 = (p16 * bottom2 + p17 * home2 + p18 * top2 + p19)
    row_pen3b = (p20 * bottom1 + p21 * home1 + p22 * top1 + p23)
    row_pen4 = (p24 * bottom2 + p25 * home2 + p26 * top2 + p27)
    row_pen4b = (p28 * bottom1 + p29 * home1 + p30 * top1 + p31)

    col_pen1 = (p32 * features[5] + p33 * features[6] + p34 * features[7] + p35 * features[8] + p36)
    col_pen2 = (p37 * features[5] + p38 * features[6] + p39 * features[7] + p40 * features[8] + p41)
    col_pen2b = (p42 * features[1] + p43 * features[2] + p44 * features[3] + p45 * features[4] + p46)
    col_pen3 = (p47 * features[5] + p48 * features[6] + p49 * features[7] + p50 * features[8] + p51)
    col_pen3b = (p52 * features[1] + p53 * features[2] + p54 * features[3] + p55 * features[4] + p56)
    col_pen4 = (p57 * features[5] + p58 * features[6] + p59 * features[7] + p60 * features[8] + p61)
    col_pen4b = (p62 * features[1] + p63 * features[2] + p64 * features[3] + p65 * features[4] + p66)

    # finger_w1 =  + p67
    dist = (dx ** 2 + dy ** 2) ** 0.5
    finger_w2 = col_pen2 * row_pen2 * row_pen2b * col_pen2b
    finger_w3 = col_pen3 * row_pen3 * row_pen3b * col_pen3b + p69
    
    sfb_pen = (p70 * is_sfb * dist + p71)
    scissor_pen = (p73 * (1 - is_sfb) * same_hand * (p74 + dy * p75) * finger_w2 + p76) # 
    alt_boost = (p77 * (1 - same_hand) * finger_w3 + p78)
    finger_pen = col_pen1 * row_pen1 + col_pen4 * col_pen4b * row_pen4 * row_pen4b

    return freq_pen * alt_boost * scissor_pen * sfb_pen * finger_pen

sfb mode
def model_function(features, a, b, c, d, e, f, g, h, i, j, k, l, m, n, o, p, q, r, t, u, v, w, x, y, z, aa, ab, ac, ad, ae, af, ag, ah, aj, ai, ak, al, am, an): # 
    freq, is_sfb, dx, dy, same_hand = features[0], features[9], features[10], features[11], features[12]
    bottom, home, top = features[13:16]
    freq_pen = (a * np.log(freq + b) + c)
    
    # sfb_finger_pen = (d*features[5]+e*features[6]+f*features[7]+g*features[8] + h)
    row_pen1 = (d*bottom+e*home+f*top+g)
    row_pen2 = (h*bottom+aa*home+ab*top+ac)
    row_pen3 = (ai*bottom+aj*home+ak*top+al)

    finger_pen1 = (t*features[5]+u*features[6]+v*features[7]+w*features[8] + x)
    finger_pen2 = (k*features[5]+l*features[6]+m*features[7]+n*features[8] + o)
    finger_pen3 = (ad*features[5]+ae*features[6]+af*features[7]+ag*features[8] + ah)
    #finger_pen = finger_pen1+finger_pen2

    dist = (dx**2+dy**2)**0.5
    sfb_pen = (i*is_sfb*dist*(finger_pen1+row_pen1)+j)
    scissor_pen = ((1-is_sfb)*p*(1/(dx+r))*(finger_pen2+row_pen2)*same_hand*(al+dy*am)+q)
    alt_boost = y*(1-same_hand)*(finger_pen3+row_pen3)+z

    return freq_pen+alt_boost+sfb_pen+scissor_pen
"""

freqs, times, is_sfb, delta_x, is_pinky1, is_ring1, is_middle1, is_index1, is_pinky2, is_ring2, is_middle2, is_index2, y_dist, same_hand, bottom1, home1, top1, bottom2, home2, top2, c = zip(*sorted(zip(freqs, times, is_sfb, delta_x, is_pinky1, is_ring1, is_middle1, is_index1, is_pinky2, is_ring2, is_middle2, is_index2, y_dist, same_hand, bottom1, home1, top1, bottom2, home2, top2, c), key = lambda x: x[0]))

freqs = np.array(freqs, dtype=np.float64)
times = np.array(times, dtype=np.float64)
is_sfb = np.array(is_sfb, dtype=np.float64)
delta_x = np.array(delta_x, dtype=np.float64)
top1 = np.array(top1, dtype=np.float64)
home1 = np.array(home1, dtype=np.float64)
bottom1 = np.array(bottom1, dtype=np.float64)
top2 = np.array(top2, dtype=np.float64)
home2 = np.array(home2, dtype=np.float64)
bottom2 = np.array(bottom2, dtype=np.float64)
finger1 = np.array(finger1, dtype=np.float64)
finger2 = np.array(finger2, dtype=np.float64)
is_pinky1 = np.array(is_pinky1, dtype=np.float64)
is_pinky2 = np.array(is_pinky2, dtype=np.float64)
is_ring1 = np.array(is_ring1, dtype=np.float64)
is_ring2 = np.array(is_ring2, dtype=np.float64)
is_middle1 = np.array(is_middle1, dtype=np.float64)
is_middle2 = np.array(is_middle2, dtype=np.float64)
is_index1 = np.array(is_index1, dtype=np.float64)
is_index2 = np.array(is_index2, dtype=np.float64)
dists = np.array(dists, dtype=np.float64)
y_dist= np.array(y_dist, dtype=np.float64)
same_hand= np.array(same_hand, dtype=np.float64)

input_data = [freqs, is_pinky1, is_ring1, is_middle1, is_index1, is_pinky2, is_ring2, is_middle2, is_index2, is_sfb, delta_x, y_dist, same_hand, bottom1, home1, top1, bottom2, home2, top2]
popt, pcov = curve_fit(model_function, input_data, times, maxfev=75000)

/tmp/ipykernel_4891/164898602.py:16: RuntimeWarning: invalid value encountered in log
  freq_pen = (p1 * np.log(freq + p2) + p3)
/usr/local/lib/python3.8/dist-packages/scipy/optimize/_minpack_py.py:906: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',


In [110]:
%matplotlib qt

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

diverging = ['#ff0055', '#ff1959', '#ff265c', '#ff305f', '#ff3963', '#ff4066', '#ff4769', '#ff4d6d', '#ff5370', '#ff5973', '#ff5e77', '#ff647a', '#ff697e', '#ff6e81', '#ff7284', '#ff7788', '#ff7c8b', '#ff808e', '#ff8592', '#ff8995', '#ff8d99', '#ff929c', '#ff96a0', '#ff9aa3', '#ff9ea6', '#ffa2aa', '#ffa6ad', '#ffaab1', '#ffaeb4', '#ffb2b8', '#ffb6bb', '#ffbabf', '#ffbec2', '#ffc2c6', '#ffc6c9', '#ffcacd', '#ffced0', '#ffd1d4', '#ffd5d7', '#ffd9db', '#ffdddf', '#ffe1e2', '#ffe5e6', '#ffe8e9', '#ffeced', '#fff0f1', '#fff4f4', '#fff7f8', '#fffbfb', '#ffffff', '#fbfdfe', '#f7fbfc', '#f3f9fb', '#eff8f9', '#ebf6f8', '#e7f4f7', '#e3f2f5', '#dff0f4', '#dbeef2', '#d7ecf1', '#d3ebf0', '#cfe9ee', '#cbe7ed', '#c7e5eb', '#c3e3ea', '#bfe1e9', '#bbdfe7', '#b7dee6', '#b3dce4', '#afdae3', '#abd8e2', '#a6d6e0', '#a2d4df', '#9ed2dd', '#9ad1dc', '#96cfda', '#91cdd9', '#8dcbd8', '#89c9d6', '#84c7d5', '#80c5d3', '#7bc3d2', '#77c2d1', '#72c0cf', '#6ebece', '#69bccc', '#64bacb', '#5fb8c9', '#5ab6c8', '#54b4c7', '#4fb2c5', '#49b1c4', '#43afc2', '#3cadc1', '#35abc0', '#2da9be', '#23a7bd', '#17a5bb', '#00a3ba']
rainbow = ['#800000', '#81000e', '#820119', '#830222', '#84042a', '#850631', '#850938', '#850c3f', '#851046', '#85134c', '#851653', '#841959', '#831c5f', '#821f65', '#81226c', '#802571', '#7f2877', '#7d2a7d', '#7b2d82', '#793088', '#77338d', '#753692', '#723997', '#6f3c9b', '#6c3fa0', '#6942a4', '#6545a7', '#6248ab', '#5e4bae', '#594eb1', '#5451b4', '#4f54b6', '#4a57b8', '#445bba', '#3d5ebb', '#3561bc', '#2b64bd', '#1f67bd', '#0b6bbd', '#006ebd', '#0071bc', '#0074bb', '#0077ba', '#007ab8', '#007db6', '#0080b4', '#0083b2', '#0085af', '#0088ac', '#008aa9', '#008da5', '#008fa2', '#00919e', '#00939a', '#009595', '#009791', '#00998c', '#009a88', '#009b83', '#009d7e', '#009e79', '#009f73', '#00a06e', '#00a068', '#00a163', '#1da15d', '#31a257', '#40a251', '#4ca24b', '#57a245', '#61a23f', '#6ba138', '#74a131', '#7ca02a', '#84a022', '#8c9f19', '#949e0d', '#9b9d00', '#a29c00', '#a99b00', '#af9a00', '#b69900', '#bc9800', '#c29700', '#c79600', '#cd9400', '#d29300', '#d79210', '#db911c', '#e09026', '#e48f2e', '#e88e37', '#ec8d3f', '#ef8c47', '#f38b4f', '#f68b56', '#f88a5e', '#fb8a66', '#fd8a6d']
magma = ['#000a1e', '#000c22', '#000d27', '#000f2b', '#001030', '#001134', '#001338', '#01143d', '#021541', '#031646', '#05174a', '#08194f', '#0b1a53', '#0e1b57', '#121c5c', '#151d60', '#191e64', '#1c1f68', '#20206d', '#242171', '#272274', '#2b2278', '#2f237c', '#332480', '#372583', '#3c2687', '#40278a', '#44288d', '#492890', '#4d2993', '#522a95', '#572b98', '#5b2c9a', '#602d9c', '#652e9e', '#6a2ea0', '#6f2fa2', '#7430a3', '#7931a4', '#7e32a5', '#8433a6', '#8934a7', '#8e35a7', '#9336a7', '#9937a7', '#9e38a7', '#a339a7', '#a83ba6', '#ae3ca6', '#b33da5', '#b83fa3', '#bd40a2', '#c242a0', '#c7439f', '#cc459d', '#d1479b', '#d64999', '#da4b96', '#df4e94', '#e35091', '#e8538e', '#ec568c', '#f05889', '#f45c85', '#f75f82', '#fb627f', '#fe667c', '#ff6a79', '#ff6e75', '#ff7272', '#ff766f', '#ff7b6b', '#ff7f68', '#ff8465', '#ff8962', '#ff8d60', '#ff925e', '#ff975c', '#ff9d5a', '#ffa259', '#ffa759', '#ffac59', '#ffb25a', '#ffb75b', '#ffbc5d', '#ffc160', '#ffc764', '#ffcc68', '#ffd16d', '#ffd672', '#ffdb78', '#ffe07f', '#ffe586', '#ffea8d', '#ffee94', '#fef39c', '#fcf7a4', '#fbfbac', '#f9ffb4']
viv = list(reversed(['#fff829', '#f4f730', '#e8f737', '#ddf63f', '#d1f546', '#c5f34d', '#b9f254', '#acf15b', '#a0ef62', '#92ed68', '#85ec6e', '#76ea74', '#66e77a', '#55e580', '#41e385', '#24e08b', '#00dd90', '#00da95', '#00d799', '#00d49e', '#00d1a2', '#00cda6', '#00caaa', '#00c6ae', '#00c2b1', '#00beb4', '#00bab7', '#00b6ba', '#00b2bc', '#00adbf', '#00a9c0', '#00a4c2', '#00a0c3', '#009bc4', '#0096c5', '#0091c6', '#008dc6', '#0088c6', '#0083c5', '#007ec4', '#0079c3', '#0074c2', '#006fc0', '#0069be', '#0064bc', '#005fb9', '#145ab6', '#2155b3', '#2950b0', '#304bac', '#3547a8', '#3942a4', '#3d3d9f', '#40389b', '#423396', '#442f91', '#462a8b', '#472586', '#482080', '#491c7a', '#491775', '#49116f', '#490c68', '#490562']))
mycmap = LinearSegmentedColormap.from_list('custom_colormap', magma)
jump = 1

plt.figure()
new_y = model_function([arr[::jump] for arr in input_data], *popt)
scatter = plt.scatter(freqs[::jump], times[::jump], c=c[::jump], s=50)

plt.plot(freqs[::jump], new_y, c="black")
#plt.axhline(0, color='black', linewidth=0.5)
#plt.scatter(freqs, times-new_y, c="red")
plt.xlabel("Number of Occurrences in Corpus ")
plt.ylabel("Average Typing Time (Milliseconds)")
plt.xscale("log")

plt.show()

sum_of_squares = np.sum((times - np.mean(times))**2)

residuals = times-new_y
print("R^2:", 1 - np.sum((residuals)**2)/sum_of_squares)
print("MAE:", np.mean(np.abs(residuals)))

# What do these errors mean?
# print("Param Errors:", np.sqrt(np.diag(pcov)))

for i, v in enumerate(popt):
    print(f"p{i}:", v)


R^2: 0.7139786454105281
MAE: 20.925918881178823
p0: -20.192306701327283
p1: -155.9836974931625
p2: 403.60819732323677
p3: -1.276031167037464
p4: -3.473889051298411
p5: 0.4193094415702458
p6: 1.0
p7: 6.712246715165405
p8: -1.3027781316534153
p9: -0.7508515905017628
p10: 1.0
p11: 0.8727831518673136
p12: -0.5674371876157627
p13: 0.9297844030493612
p14: 3.0592982191265734
p15: 2.368023663481908
p16: 3.1466155373809173
p17: 2.739392231135773
p18: -0.4720541372314334
p19: 27.130554809578836
p20: 1.0
p21: 52.01791901534216


In [6]:
import numpy as np
import matplotlib.pyplot as plt

# Plotting the covariance matrix as a heatmap
print(pcov)

# sorted_pcov = pcov[np.argsort(np.sum(np.clip(pcov, 0, None), axis=0))]

def symmetric_log_transform(matrix):
    sign_matrix = np.sign(matrix)
    log_matrix = np.log(np.abs(matrix)*0.01 + 1)
    return sign_matrix * log_matrix

plt.figure(figsize=(6, 6))
plt.imshow(symmetric_log_transform(pcov), cmap=mycmap, interpolation='nearest')
plt.colorbar(label='Covariance')
plt.title('Normalized Covariance Matrix')
plt.xticks(np.arange(pcov.shape[0], step=1), np.arange(pcov.shape[0], step=1))
plt.yticks(np.arange(pcov.shape[1], step=1), np.arange(pcov.shape[1], step=1))
plt.xlabel('Parameters')
plt.ylabel('Parameters')
plt.show()

[[ 1.27964030e+00 -4.75264629e+02 -1.60111620e+01]
 [-4.75264629e+02  7.01616843e+05  6.27814732e+03]
 [-1.60111620e+01  6.27814732e+03  2.03701252e+02]]
